# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [1]:
import pandas as pd
import numpy as np

#  Load Data 
file_urls = [
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv",
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv",
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"
]

# Load all three files into a list of DataFrames
dataframes = [pd.read_csv(url) for url in file_urls]

# Define Cleaning Function

def clean_data(df):
    """
    Performs standard cleaning and formatting on the customer data.
    """
    
    #  Standardize Column Names (convert to snake_case)
    df.columns = df.columns.str.lower().str.replace(' ', '_', regex=False)
    
    # Drop unnecessary columns (often old index columns from prior saving)
    df = df.drop(columns=['unnamed:_0', 'customer'], errors='ignore')
    
    #  Clean and Convert Numeric/Currency Columns
    
    # Clean 'customer_lifetime_value'
    if 'customer_lifetime_value' in df.columns:
        # Remove all non-numeric characters except the decimal point
        df['customer_lifetime_value'] = (
            df['customer_lifetime_value']
            .astype(str)
            .str.replace('[^0-9.]', '', regex=True)
        )
        # Convert to numeric, coercing errors to NaN
        df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')

    # Clean 'income'
    if 'income' in df.columns:
        # Simply coerce to numeric (removes commas if present and handles strings)
        df['income'] = pd.to_numeric(df['income'], errors='coerce')
        
    # Clean and Standardize String Columns
    
    # Standardize 'gender' (f/m to female/male)
    if 'gender' in df.columns:
        df['gender'] = (
            df['gender']
            .astype(str).str.lower().str.strip()
            .replace({'f': 'female', 'm': 'male'}, regex=False)
            .replace('nan', pd.NA) # Replace the string 'nan' with a proper NA value
        )

    # Standardize all other object (string) columns to Title Case and handle 'NaN' string
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.title().replace('Nan', pd.NA)
        
    #  Handle Missing Values and Duplicates
    
    # Drop rows where all non-ID/Index columns are NaN
    df = df.dropna(how='all') 
    
    # Drop duplicate rows
    df = df.drop_duplicates()

    return df

# Clean and Combine Data 

# Apply the cleaning function to each DataFrame
clean_dataframes = [clean_data(df) for df in dataframes]

# Concatenate all cleaned DataFrames into one
combined_data = pd.concat(clean_dataframes, ignore_index=True)

# Verification 
print(f"Shape of Combined Data: {combined_data.shape}")
print("\nCombined Data Info:")
combined_data.info()
print("\nFirst 5 rows of Combined Data:")
print(combined_data.head())

Shape of Combined Data: (8888, 11)

Combined Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8888 entries, 0 to 8887
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   st                         2060 non-null   object 
 1   gender                     8888 non-null   object 
 2   education                  8887 non-null   object 
 3   customer_lifetime_value    8880 non-null   float64
 4   income                     8887 non-null   float64
 5   monthly_premium_auto       8887 non-null   float64
 6   number_of_open_complaints  8887 non-null   object 
 7   policy_type                8887 non-null   object 
 8   vehicle_class              8887 non-null   object 
 9   total_claim_amount         8887 non-null   float64
 10  state                      6827 non-null   object 
dtypes: float64(4), object(7)
memory usage: 763.9+ KB

First 5 rows of Combined Data:
           st  gender  

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [2]:
import pandas as pd
import numpy as np

# Load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df = pd.read_csv(url)

# Display a quick preview to confirm column names
# print(df.columns)
# print(df['Sales Channel'].unique())

# Standardize column names for easier access (if necessary, though they look clean)
df.columns = df.columns.str.lower().str.replace(' ', '_')

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [3]:
# Create the pivot table for total revenue (CLV) by Sales Channel
revenue_by_channel = pd.pivot_table(
    df, 
    values='customer_lifetime_value', 
    index='sales_channel', 
    aggfunc='sum'
)

# Rename the column and round to 2 decimal places
revenue_by_channel = revenue_by_channel.rename(
    columns={'customer_lifetime_value': 'Total Revenue (CLV)'}
).round(2)

print("Total Revenue (CLV) by Sales Channel:")
print(revenue_by_channel.sort_values(by='Total Revenue (CLV)', ascending=False))

Total Revenue (CLV) by Sales Channel:
               Total Revenue (CLV)
sales_channel                     
Agent                  33057887.85
Branch                 24359201.21
Call Center            17364288.37
Web                    12697632.90


In [4]:
# Create the pivot table for average CLV by Gender and Education
avg_clv_by_demographics = pd.pivot_table(
    df, 
    values='customer_lifetime_value', 
    index='gender', 
    columns='education', 
    aggfunc='mean'
).round(2)

print("\nAverage Customer Lifetime Value (CLV) by Gender and Education:")
print(avg_clv_by_demographics)


Average Customer Lifetime Value (CLV) by Gender and Education:
education  Bachelor  College   Doctor  High School or Below   Master
gender                                                              
F           7874.27  7748.82  7328.51               8675.22  8157.05
M           7703.60  8052.46  7415.33               8149.69  8168.83


## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here